# 02 — Entrenamiento (Kaggle / Colab)

Entrena en una GPU de la nube los modelos de las dos primeras piezas del pipeline, para agua y para energía:

- **Segmentación del visor**: YOLOv8n-seg (Ultralytics) y Mask R-CNN (torchvision).
- **Detección de dígitos**: YOLOv8n y YOLO11n sobre el visor rectificado.

Las otras piezas no necesitan GPU y se entrenan o evalúan en local (ver la sección 5): el clasificador de formato con `scripts/train_meter_model.py` y la lectura de punta a punta con `scripts/evaluate_pipeline.py`. Los resultados de todo están en `03_resultados.ipynb` y el detalle en [experiments/README.md](../experiments/README.md).

El notebook corre de punta a punta los experimentos de la lista `EXPERIMENTS`: clona el repositorio, descarga y prepara solo los datos que esos experimentos necesitan, entrena, evalúa y empaqueta los resultados (`models/` y `experiments/`) en un zip por experimento.

**Kaggle:** si algún experimento usa Toloka (`toloka`, `combined` o `combined_energia`), adjuntar ese dataset con *Add Input* (buscar "yandextoloka water meters"; una ejecución en segundo plano no puede adjuntarlo sola). Activar *GPU* e *Internet* en *Settings*, cargar el Secret `ROBOFLOW_API_KEY` (*Add-ons → Secrets*) y ejecutar con *Save Version → Save & Run All* para que corra en segundo plano.

**Colab:** *Entorno de ejecución → GPU* y cargar el Secret `ROBOFLOW_API_KEY`.

## 1. Experimentos

Cada experimento es un diccionario con el nombre (que es también el nombre del modelo en `models/` y de la carpeta en `experiments/`), la pieza que entrena (`task`), los datos (`dataset`), el modelo base preentrenado y los hiperparámetros. Los que ya se corrieron quedan comentados, con su configuración, para poder repetirlos.

| ID | Experimento | Pieza | Servicio | Qué se prueba |
|---|---|---|---|---|
| A | `toloka_yolov8n_seg` | visor | agua | Línea de base: solo Toloka |
| B | `toloka_yfzw4_yolov8n_seg` | visor | agua | Toloka + visor derivado de water-meter-dataset (**modelo de agua**) |
| C | `toloka_yfzw4_yolov8n_seg_1024` | visor | agua | B a 1024 px |
| S4 | `toloka_yfzw4_maskrcnn` | visor | agua | Mask R-CNN R50-FPN con los datos de B |
| D1 | `digits_yolov8n` | dígitos | agua | YOLOv8n sobre water-meter-dataset (**modelo de agua**) |
| D2 | `digits_yolo11n` | dígitos | agua | YOLO11n con los mismos datos |
| E1 | `digits_mix_yolov8n` | dígitos | energía | D1 + dígitos LCD públicos (5srka y frvqn) |
| E2 | `toloka_yfzw4_energia_yolov8n_seg` | visor | energía | B + 587 visores de energía CEML (**modelo de energía**) |
| E3 | `digits_mix_energia_yolov8n` | dígitos | energía | E1 + 483 visores de energía CEML etiquetados (**modelo de energía**) |
| E4 | `digits_mix_energia_rev_yolov8n` | dígitos | energía | E3 solo con los 209 recortes revisados como legibles |
| D3 | `digits_yolov8s` | dígitos | agua | D1 con YOLOv8s (tamaño del modelo) |
| D4 | `digits_yolov8m` | dígitos | agua | D1 con YOLOv8m (tamaño del modelo) |
| D5 | `digits_yolov8s_640` | dígitos | agua | D3 a 640 px (resolución de entrada) |
| D6 | `digits_yolov8n_aug` | dígitos | agua | D1 con más aumentaciones de color y geometría |
| D7 | `digits_rtdetr_l` | dígitos | agua | RT-DETR-L (otra familia de arquitectura) |

**Hiperparámetros.** Las comparaciones cambian una sola cosa por vez: los segmentadores YOLO que se comparan (B, C y E2) usan 40 épocas, paciencia 10 y batch 16, a 640 px salvo C (A, la primera línea de base, se entrenó con 100 épocas y paciencia 20); todos los detectores de dígitos, 100 épocas, paciencia 20, batch 32 y 416 px. Los dígitos se entrenan con `fliplr=0` porque un dígito espejado puede parecer otro (2 y 5); el visor sí admite espejado.

**Ronda D3-D7 (agua, dígitos).** Cada una cambia una sola cosa respecto de D1 (excepto D5, que se compara contra D3) y se elige por el mAP50-95 de validación de water-meter-dataset; el split de prueba *público* y la latencia en CPU se informan como referencia. La elección no usa CEML ni el test; las fotos CEML se miden una sola vez al final, con `scripts/evaluate_pipeline.py`, para no seguir sesgando la prueba.

**Orden.** Cada experimento se empaqueta al terminar, así un corte por tiempo no pierde los anteriores: primero van los rápidos y al final Mask R-CNN, que es el más largo.

In [1]:
# Experimentos a ejecutar, en orden (ver experiments/README.md).
# trainer: "yolo" (Ultralytics, por defecto), "maskrcnn" (torchvision) o "eval_coco"
#          (solo evalúa con métricas COCO un modelo ya versionado en models/).
# task: "segment" (etapa 1, visor) o "detect" (etapa 2, dígitos).
# dataset: "toloka" (solo Toloka), "combined" (Toloka + yfzw4 con visor derivado)
#          "digits" (dígitos de yfzw4 sobre el visor rectificado)
#          "digits_mix" (yfzw4 + dígitos LCD de energía: meter-digits-5srka y meter-digits-frvqn)
#          "combined_energia" (combined + visores de energía CEML revisados, sin la muestra de test)
#          "digits_mix_energia" (digits_mix + dígitos de energía CEML corregidos, sin los medidores de la muestra)
#          o "digits_mix_energia_rev" (igual, sin los recortes marcados ilegibles en la revisión).
# fliplr: probabilidad de espejado horizontal; 0 para dígitos (un 2 espejado parece un 5).
# Cada experimento se empaqueta al terminar, así un corte por tiempo no pierde los anteriores:
# por eso van primero los rápidos y al final Mask R-CNN, que es el más largo.
# extra: hiperparámetros adicionales de Ultralytics (se pasan como --extra CLAVE=VALOR).
# Ronda de dígitos de agua (dataset "digits"): mismo protocolo que D1 (seed 42, 100 épocas, paciencia 20),
# cambiando una cosa por vez respecto de D1 (YOLOv8n, 416 px). Se elige con la validación PÚBLICA de
# water-meter-dataset (no con CEML) y CEML se mide una sola vez al final. Van primero los rápidos.
EXPERIMENTS = [
    # D8: RT-DETR-L con su propia receta de optimización (AdamW, lr 1e-4, weight decay 1e-4, como el
    # RT-DETR original), en lugar de la de YOLO que usó D7 (MuSGD, lr 0,01, warmup de bias en 0,1).
    # Mismos datos, resolución y aumentaciones que D1/D7. Unos 300 s por época en una T4: 100 épocas
    # entran en las 12 h de una sesión de Kaggle.
    {"name": "digits_rtdetr_l_adamw", "task": "detect", "dataset": "digits", "model": "rtdetr-l.pt",
     "imgsz": 416, "epochs": 100, "patience": 30, "batch": 16, "fliplr": 0.0,
     "extra": {"optimizer": "AdamW", "lr0": 0.0001, "weight_decay": 0.0001, "warmup_bias_lr": 0.0}},
]

# Si algún D3-D8 mejora a D1, repetirlo con los datos de energía (mismo "dataset" que E3):
# {"name": "digits_mix_energia_yolov8s", "task": "detect", "dataset": "digits_mix_energia", "model": "yolov8s.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},

# Ya ejecutados (resultados en experiments/README.md):
# {"name": "digits_rtdetr_l", "task": "detect", "dataset": "digits", "model": "rtdetr-l.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 16, "fliplr": 0.0},      # D7
# {"name": "digits_yolov8n_aug", "task": "detect", "dataset": "digits", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0,
#  "extra": {"hsv_h": 0.02, "hsv_s": 0.7, "hsv_v": 0.5, "degrees": 5.0, "perspective": 0.0005}},  # D6
# {"name": "digits_yolov8s", "task": "detect", "dataset": "digits", "model": "yolov8s.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D3
# {"name": "digits_yolov8s_640", "task": "detect", "dataset": "digits", "model": "yolov8s.pt",
#  "imgsz": 640, "epochs": 100, "patience": 20, "batch": 16, "fliplr": 0.0},      # D5
# {"name": "digits_yolov8m", "task": "detect", "dataset": "digits", "model": "yolov8m.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D4
# {"name": "digits_mix_energia_yolov8n", "task": "detect", "dataset": "digits_mix_energia", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # E3
# {"name": "digits_mix_energia_rev_yolov8n", "task": "detect", "dataset": "digits_mix_energia_rev",
#  "model": "yolov8n.pt", "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},  # E4
# {"name": "toloka_yfzw4_energia_yolov8n_seg", "task": "segment", "dataset": "combined_energia",
#  "model": "yolov8n-seg.pt", "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},  # E2
# {"name": "digits_mix_yolov8n", "task": "detect", "dataset": "digits_mix", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # E1
# {"name": "toloka_yfzw4_yolov8n_seg", "trainer": "eval_coco", "task": "segment", "dataset": "combined",
#  "imgsz": 640},                                                                  # B con métrica COCO
# {"name": "digits_yolo11n", "task": "detect", "dataset": "digits", "model": "yolo11n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D2
# {"name": "toloka_yfzw4_maskrcnn", "trainer": "maskrcnn", "task": "segment", "dataset": "combined",
#  "epochs": 10, "batch": 4, "lr": 0.005},                                         # S4
# {"name": "toloka_yfzw4_yolov8n_seg", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 640, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},       # B
# {"name": "toloka_yfzw4_yolov8n_seg_1024", "task": "segment", "dataset": "combined", "model": "yolov8n-seg.pt",
#  "imgsz": 1024, "epochs": 40, "patience": 10, "batch": 16, "fliplr": 0.5},      # C
# {"name": "digits_yolov8n", "task": "detect", "dataset": "digits", "model": "yolov8n.pt",
#  "imgsz": 416, "epochs": 100, "patience": 20, "batch": 32, "fliplr": 0.0},      # D1

## 2. Entorno

Detecta si corre en Kaggle, en Colab o en local. En la nube clona el repositorio (rama `REPO_BRANCH`), instala solo lo que falta para entrenar (PyTorch y torchvision con CUDA ya vienen instalados) y lee la API key de Roboflow de los Secrets. En local trabaja desde la raíz del repo con el entorno ya instalado.

In [2]:
# Configuración del entorno: Kaggle, Colab o local.
import os
import subprocess
import sys
from pathlib import Path

IN_KAGGLE = Path("/kaggle/working").exists()
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/martinlacheski/ceia-vpc-2.git"
REPO_BRANCH = "main"


def run(cmd: list[str]) -> None:
    """Ejecuta un comando mostrando su salida y falla si termina con error."""
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if IN_KAGGLE or IN_COLAB:
    os.chdir("/kaggle/working" if IN_KAGGLE else "/content")
    if not Path("ceia-vpc-2").exists():
        run(["git", "clone", "-q", "-b", REPO_BRANCH, REPO_URL])
    os.chdir("ceia-vpc-2")
    # Solo lo necesario para entrenar: PyTorch y torchvision con CUDA ya vienen instalados.
    # ultralytics fijo en la versión de requirements.txt (la misma que se verificó con RT-DETR).
    run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.212", "roboflow", "kagglehub", "opencv-python-headless",
                 "pycocotools"])

    try:
        if IN_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            os.environ["ROBOFLOW_API_KEY"] = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        else:
            from google.colab import userdata
            os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
    except Exception as exc:
        print("Aviso: no se pudo leer el Secret ROBOFLOW_API_KEY:", exc)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

import torch
print("Directorio:", Path.cwd(), "| Kaggle:", IN_KAGGLE, "| Colab:", IN_COLAB)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no disponible")

$ git clone -q -b main https://github.com/martinlacheski/ceia-vpc-2.git
$ /usr/bin/python3 -m pip install -q ultralytics==8.4.212 roboflow kagglehub opencv-python-headless pycocotools


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


Directorio: /kaggle/working/ceia-vpc-2 | Kaggle: True | Colab: False
GPU: Tesla T4


## 3. Datos

Descarga y prepara solo los datos que piden los experimentos de la lista (todos los scripts están en `scripts/` y no descargan de nuevo lo que ya está):

- **`toloka`**: Toloka con su polígono del visor, en formato YOLO-seg (`prepare_toloka.py`).
- **`combined`**: Toloka + water-meter-dataset, con el visor derivado de las cajas de los dígitos (`prepare_seg_combined.py`). Valida y prueba con Toloka.
- **`combined_energia`**: `combined` + los visores de energía CEML revisados. Suma a train solo las fotos de medidores que no están en la muestra de prueba; val y test siguen siendo los de Toloka, así las métricas se comparan con B.
- **`digits`**: cada foto de water-meter-dataset recortada al visor y rectificada igual que en el pipeline, con las cajas de los dígitos transformadas al recorte (`prepare_digits.py`). Así el detector ve el mismo encuadre que en la lectura.
- **`digits_mix`**: `digits` + los dígitos LCD públicos (5srka y frvqn).
- **`digits_mix_energia`**: `digits_mix` + los visores de energía CEML etiquetados (`--crops`), sin las fotos de los medidores de la muestra. El split train/val es por medidor, para que el mismo medidor no quede en los dos lados.
- **`digits_mix_energia_rev`**: igual, sin los recortes que la revisión marcó como ilegibles (`--skip-unreadable`).

Toloka y water-meter-dataset se descargan con `download_data.py` (Toloka necesita estar adjunto en Kaggle); los LCD públicos, con `--only energia`. Las fotos de energía CEML ya están en el repositorio (`data/ceml_energia/`).

In [3]:
# Descarga y preparación de los datos (solo los que piden los experimentos).
datasets = {e["dataset"] for e in EXPERIMENTS}
needs_toloka = bool(datasets & {"toloka", "combined", "combined_energia"})
digit_mixes = {"digits_mix", "digits_mix_energia", "digits_mix_energia_rev"}
needs_yfzw4 = bool(datasets & ({"combined", "combined_energia", "digits"} | digit_mixes))
if needs_toloka and needs_yfzw4:
    run([sys.executable, "scripts/download_data.py"])
elif needs_toloka:
    run([sys.executable, "scripts/download_data.py", "--only", "toloka"])
else:
    run([sys.executable, "scripts/download_data.py", "--only", "roboflow"])

if datasets & digit_mixes:
    run([sys.executable, "scripts/download_data.py", "--only", "energia"])

if needs_toloka:
    run([sys.executable, "scripts/prepare_toloka.py"])
if "combined" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py"])
if "combined_energia" in datasets:
    run([sys.executable, "scripts/prepare_seg_combined.py", "--extra", "data/ceml_energia",
         "--out", "data/processed/seg_toloka_yfzw4_energia"])
if "digits" in datasets:
    run([sys.executable, "scripts/prepare_digits.py"])
if "digits_mix" in datasets:
    run([sys.executable, "scripts/prepare_digits.py", "--src", "data/raw/roboflow_water_meter",
         "--src", "data/raw/energia/meter-digits-5srka", "--src", "data/raw/energia/meter-digits-frvqn",
         "--out", "data/processed/digits_mix"])
if "digits_mix_energia" in datasets:
    run([sys.executable, "scripts/prepare_digits.py", "--src", "data/raw/roboflow_water_meter",
         "--src", "data/raw/energia/meter-digits-5srka", "--src", "data/raw/energia/meter-digits-frvqn",
         "--crops", "data/ceml_energia", "--out", "data/processed/digits_mix_energia"])
if "digits_mix_energia_rev" in datasets:
    run([sys.executable, "scripts/prepare_digits.py", "--src", "data/raw/roboflow_water_meter",
         "--src", "data/raw/energia/meter-digits-5srka", "--src", "data/raw/energia/meter-digits-frvqn",
         "--crops", "data/ceml_energia", "--skip-unreadable", "--out", "data/processed/digits_mix_energia_rev"])

DATA = {
    "toloka": "data/processed/toloka_seg/data.yaml",
    "combined": "data/processed/seg_toloka_yfzw4/data.yaml",
    "combined_energia": "data/processed/seg_toloka_yfzw4_energia/data.yaml",
    "digits": "data/processed/digits_yfzw4/data.yaml",
    "digits_mix": "data/processed/digits_mix/data.yaml",
    "digits_mix_energia": "data/processed/digits_mix_energia/data.yaml",
    "digits_mix_energia_rev": "data/processed/digits_mix_energia_rev/data.yaml",
}

$ /usr/bin/python3 scripts/download_data.py --only roboflow
loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to data/raw/roboflow_water_meter in yolov8::   0%|          | 0/17968 [00:00<?, ?it/s]

Extracting Dataset Version Zip to data/raw/roboflow_water_meter in yolov8:: 100%|██████████| 17968/17968 [00:02<00:00, 7378.00it/s] 


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
water-meter-dataset v3 descargado en data/raw/roboflow_water_meter
$ /usr/bin/python3 scripts/prepare_digits.py
Recortes por split: {'train': 6848, 'val': 719, 'test': 656}


## 4. Entrenamiento, evaluación y empaquetado

Para cada experimento:

1. **Entrena** con `scripts/train_yolo.py` (Ultralytics) o `scripts/train_maskrcnn.py` (torchvision). Los pesos finales quedan en `models/<nombre>.pt` y las curvas y gráficos de Ultralytics en `experiments/<nombre>/`.
2. **Evalúa**:
   - segmentadores: `scripts/evaluate_seg.py` en el test de Toloka y en las fotos CEML de agua (dominio real), con la latencia en CPU; `scripts/evaluate_seg_coco.py` cuando se compara contra Mask R-CNN, porque las dos librerías no calculan el mAP igual;
   - detectores de dígitos: el split de prueba de water-meter-dataset (más los LCD públicos en los `digits_mix`), con el mAP por clase.
   Las métricas quedan en `experiments/<nombre>/eval.json`.
3. **Empaqueta** pesos y `experiments/<nombre>/` en `<nombre>.zip` (en Kaggle, en la pestaña *Output*).

In [4]:
# Entrenamiento, evaluación y empaquetado de cada experimento.
import json
import shutil

out_dir = Path("/kaggle/working") if IN_KAGGLE else Path.cwd()


def package(name: str, weights: list[str]) -> str:
    """Arma <name>.zip con los pesos y experiments/<name>/ (en Kaggle: pestaña Output)."""
    staging = Path("runs/package") / name
    shutil.rmtree(staging, ignore_errors=True)
    (staging / "models").mkdir(parents=True)
    for path in weights:
        shutil.copy2(path, staging / "models")
    shutil.copytree(f"experiments/{name}", staging / "experiments" / name)
    return shutil.make_archive(str(out_dir / name), "zip", root_dir=staging)


for exp in EXPERIMENTS:
    print(f"\n===== {exp['name']} =====")
    trainer = exp.get("trainer", "yolo")
    if trainer == "eval_coco":
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "yolo",
             "--weights", f"models/{exp['name']}.pt", "--imgsz", str(exp["imgsz"])])
        print("Resultados:", package(exp["name"], []))
        continue
    if trainer == "maskrcnn":
        root = str(Path(DATA[exp["dataset"]]).parent)
        run([sys.executable, "scripts/train_maskrcnn.py", "--data", root, "--name", exp["name"],
             "--epochs", str(exp["epochs"]), "--batch", str(exp["batch"]), "--lr", str(exp["lr"])])
        run([sys.executable, "scripts/evaluate_seg_coco.py", "--arch", "maskrcnn",
             "--weights", f"models/{exp['name']}.pth"])
        print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pth"]))
        continue

    run([sys.executable, "scripts/train_yolo.py", "--task", exp["task"], "--model", exp["model"],
         "--data", DATA[exp["dataset"]], "--name", exp["name"], "--imgsz", str(exp["imgsz"]),
         "--epochs", str(exp["epochs"]), "--patience", str(exp["patience"]), "--batch", str(exp["batch"]),
         "--fliplr", str(exp["fliplr"])]
        + [arg for key, value in exp.get("extra", {}).items() for arg in ("--extra", f"{key}={value!r}")])
    if exp["task"] == "segment":
        run([sys.executable, "scripts/evaluate_seg.py", "--model", f"models/{exp['name']}.pt",
             "--imgsz", str(exp["imgsz"])])
    else:
        # Dígitos: métricas sobre el split de prueba de yfzw4, globales y por clase.
        from ultralytics import YOLO
        res = YOLO(f"models/{exp['name']}.pt").val(data=DATA[exp["dataset"]], split="test",
                                                   imgsz=exp["imgsz"], verbose=False)
        evaluation = {
            "test": {"map50": round(float(res.box.map50), 4), "map50_95": round(float(res.box.map), 4),
                     "precision": round(float(res.box.mp), 4), "recall": round(float(res.box.mr), 4)},
            "map50_95_por_clase": {res.names[int(c)]: round(float(res.box.maps[int(c)]), 4)
                                    for c in res.box.ap_class_index},
            "imgsz": exp["imgsz"],
        }
        # Latencia en CPU (ms por imagen) sobre 30 recortes del split de prueba; depende de la máquina.
        import time
        test_dir = Path(DATA[exp["dataset"]]).parent / "images" / "test"
        imgs = sorted(str(p) for p in test_dir.iterdir())[:30]
        model = YOLO(f"models/{exp['name']}.pt")
        model.predict(imgs[:3], device="cpu", imgsz=exp["imgsz"], verbose=False)  # calentamiento
        t0 = time.perf_counter()
        model.predict(imgs, device="cpu", imgsz=exp["imgsz"], verbose=False)
        evaluation["latencia_cpu_ms"] = round((time.perf_counter() - t0) / len(imgs) * 1000, 1)
        evaluation["pesos_mb"] = round(Path(f"models/{exp['name']}.pt").stat().st_size / 1e6, 1)
        Path(f"experiments/{exp['name']}/eval.json").write_text(json.dumps(evaluation, indent=2))
        print(json.dumps(evaluation, indent=2))
    print("Resultados:", package(exp["name"], [f"models/{exp['name']}.pt"]))


===== digits_rtdetr_l_adamw =====
$ /usr/bin/python3 scripts/train_yolo.py --task detect --model rtdetr-l.pt --data data/processed/digits_yfzw4/data.yaml --name digits_rtdetr_l_adamw --imgsz 416 --epochs 100 --patience 30 --batch 16 --fliplr 0.0 --extra optimizer='AdamW' --extra lr0=0.0001 --extra weight_decay=0.0001 --extra warmup_bias_lr=0.0
New https://pypi.org/project/ultralytics/8.4.218 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.212 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/processed/digits_yfzw4/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      1/100      7.68G      0.933      2.999     0.7697        218        416: 100% ━━━━━━━━━━━━ 428/428 1.8it/s 3:58
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.6it/s 8.8s
                   all        719       4463     0.0594      0.571     0.0718     0.0317


/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(



      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


... (147 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     76/100      7.16G     0.2636     0.4693      0.198        213        416: 100% ━━━━━━━━━━━━ 428/428 1.8it/s 3:54
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.2s
                   all        719       4463      0.975      0.951      0.972      0.587

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size


/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     77/100      7.16G     0.2649     0.4701     0.1993        237        416: 100% ━━━━━━━━━━━━ 428/428 1.8it/s 3:55
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 23/23 2.8it/s 8.3s
                   all        719       4463      0.976      0.948      0.971      0.586
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 47, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

77 epochs completed in 5.192 hours.
Optimizer stripped from /kaggle/working/ceia-vpc-2/runs/detect/digits_rtdetr_l_adamw/weights/last.pt, 66.2MB
Optimizer stripped from /kaggle/working/ceia-vpc-2/runs/detect/digits_rtdetr_l_adamw/weights/best.pt, 66.2MB

Validating /kaggle/working/ceia-vpc-2/runs/detect/digits_rtdetr_l_adamw/weights/best.pt...
Ultralytics 8.4.212 🚀 Python-3.13.16 torc

## 5. Después de entrenar

1. Descargar los zips de *Output* y descomprimirlos en la raíz del repositorio: cada uno trae `models/<nombre>.pt` y `experiments/<nombre>/`.
2. **Clasificador de formato** (en local, CPU o GPU chica; unos minutos):

   ```bash
   python scripts/train_meter_model.py --name meter_layout_mnv3s --target formato           # agua
   python scripts/train_meter_model.py --name meter_layout_agua_energia --target formato \
       --extra-data data/ceml_energia                                                       # agua + energía
   ```

3. **Lectura de punta a punta** sobre las fotos de prueba de cada servicio:

   ```bash
   python scripts/evaluate_pipeline.py                                                      # agua
   python scripts/evaluate_pipeline.py --no-flip --sheet data/ceml_energia/readings.csv \
       --images data/ceml_energia/images --seg-weights models/toloka_yfzw4_energia_yolov8n_seg.pt \
       --digits-v8 models/digits_mix_energia_yolov8n.pt --classifier meter_layout_agua_energia \
       --out experiments/pipeline_ceml_energia                                              # energía
   ```

4. Ejecutar `03_resultados.ipynb` para ver todo junto.